In [ ]:
!pip install modelscope


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 119.0 MB/s eta 0:00:00


In [ ]:
!pip -q install -U pip
!pip -q install -U \
  "torch>=2.0.0" \
  "transformers>=4.40.0" \
  "datasets>=2.14.0" \
  "accelerate>=0.26.0" \
  "peft>=0.8.0" \
  "trl>=0.7.0" \
  "bitsandbytes>=0.41.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.5 MB/s eta 0:00:00


In [ ]:
!pip -q install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os

# Use ModelScope instead of HuggingFace
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

# NOW import unsloth
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import os

# Prevent HuggingFace timeout errors
os.environ['UNSLOTH_SKIP_STATISTICS'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'

print(" Timeout prevention enabled")

 Timeout prevention enabled


In [ ]:
def mount_google_drive():
    """Mount Google Drive for saving outputs"""
    try:
        from google.colab import drive

        print("\n" + "=" * 80)
        print(" MOUNTING GOOGLE DRIVE")
        print("=" * 80)

        drive.mount('/content/drive')

        print("\n Google Drive mounted successfully!")
        print("Drive path: /content/drive/MyDrive/")

        return True

    except ImportError:
        print("\n Not running on Google Colab - using local storage")
        return False
    except Exception as e:
        print(f"\n  Error mounting Drive: {e}")
        print("Continuing with local storage...")
        return False


# Auto-detect and mount
IS_COLAB = mount_google_drive()



 MOUNTING GOOGLE DRIVE
Mounted at /content/drive

 Google Drive mounted successfully!
Drive path: /content/drive/MyDrive/


In [ ]:
import os
import json
import torch
from datasets import Dataset, load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================

# HuggingFace Token - PASTE YOUR TOKEN HERE
HF_TOKEN = os.getenv("HF_TOKEN")  # Set your Hugging Face token as an environment variable

# Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_SEQ_LENGTH = 2048  # Qwen supports up to 32k, but 2k is enough for emails
LOAD_IN_4BIT = True    # Use 4-bit quantization to reduce memory

# LoRA Configuration (for parameter-efficient fine-tuning)
LORA_R = 16            # LoRA rank (higher = more parameters but better quality)
LORA_ALPHA = 16        # LoRA alpha (usually same as rank)
LORA_DROPOUT = 0.05    # Dropout for LoRA layers
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]  # Qwen-specific modules
if IS_COLAB:
    # Google Drive paths (Colab)
    OUTPUT_DIR = "/content/drive/MyDrive/qwen-meeting-extraction"
    BASE_DATA_DIR = "/content/drive/MyDrive/meeting-data"

    print(f"\n Using Google Drive:")
    print(f"   Output: {OUTPUT_DIR}")
    print(f"   Data:   {BASE_DATA_DIR}")
else:
    # Local paths (non-Colab)
    OUTPUT_DIR = "./qwen-meeting-extraction"
    BASE_DATA_DIR = "./data"

    print(f"\nUsing local storage:")
    print(f"   Output: {OUTPUT_DIR}")
    print(f"   Data:   {BASE_DATA_DIR}")

# Data files
TRAIN_FILE = f"{BASE_DATA_DIR}/train.jsonl"
VAL_FILE = f"{BASE_DATA_DIR}/val.jsonl"
TEST_FILE = f"{BASE_DATA_DIR}/test.jsonl"


 Using Google Drive:
   Output: /content/drive/MyDrive/qwen-meeting-extraction
   Data:   /content/drive/MyDrive/meeting-data


In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n Output directory ready: {OUTPUT_DIR}")

# Training Configuration
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10
SAVE_STEPS = 50
LOGGING_STEPS = 10



 Output directory ready: /content/drive/MyDrive/qwen-meeting-extraction


In [ ]:
PRODUCTION_SYSTEM_PROMPT = """You are a specialized meeting information extraction system.

TASK: Analyze emails and extract structured meeting details with high precision.

OUTPUT FORMAT (JSON only, no explanations):
{
  "is_meeting": boolean,
  "title": string or null,
  "attendees": array of email addresses,
  "start_time": ISO 8601 string or null,
  "end_time": ISO 8601 string or null,
  "location": string or null,
  "time_confidence": "high" | "medium" | "low" | "none"
}

EXTRACTION RULES:

1. MEETING DETECTION (is_meeting):
   - TRUE if email discusses scheduling/planning a meeting, call, discussion, sync, or appointment
   - TRUE if contains meeting invitations, calendar entries, or time/location details
   - FALSE for: announcements, reports, questions, general updates, FYIs
   - Keywords: meeting, schedule, calendar, invite, call, discussion, sync, appointment

2. TITLE EXTRACTION:
   - Extract meeting subject/purpose (max 100 characters)
   - Use email subject line if it describes the meeting
   - If subject unclear, extract from body (e.g., "Let's discuss Q4 budget" → "Q4 budget discussion")
   - Keep concise: "Team Sync", "Q1 Planning Meeting", "Product Review"
   - Set to null for non-meetings

3. ATTENDEES EXTRACTION (CRITICAL - Must be email addresses):
   - Extract ALL email addresses mentioned in the email
   - Include: To, Cc, mentioned in body, forwarded lists
   - Format: lowercase, exact email addresses only
   - Example: ["john.doe@company.com", "sarah@example.org"]
   - Empty array [] for non-meetings or if no emails found

4. START_TIME EXTRACTION:
   - Convert ALL times to ISO 8601 UTC format: "YYYY-MM-DDTHH:MM:SSZ"
   - Parse formats: "March 15 at 2pm", "tomorrow 3:00 PM", "Monday at 9am"
   - If date missing, infer from context (e.g., "tomorrow" relative to email date if available)
   - If only time given (e.g., "3pm"), use a reasonable date placeholder
   - Set to null if no time mentioned or non-meeting

5. END_TIME EXTRACTION:
   - Same format as start_time: "YYYY-MM-DDTHH:MM:SSZ"
   - Extract if explicitly mentioned (e.g., "2pm-3pm", "from 9am to 10am")
   - Set to null if not specified or non-meeting

6. LOCATION EXTRACTION:
   - Extract specific locations: room numbers, addresses, virtual platforms
   - Examples: "Room 301", "EB3270", "Zoom", "Google Meet", "Conference Room A"
   - Include meeting links if mentioned (e.g., "Zoom: https://...")
   - Prefer specific over generic: "Room 301" > "office"
   - Set to null if not mentioned or non-meeting

7. TIME_CONFIDENCE:
   - "high": Exact date and time specified ("March 15, 2024 at 2:00 PM")
   - "medium": Approximate time ("tomorrow afternoon", "next Monday at 3pm")
   - "low": Vague timing ("next week", "sometime soon")
   - "none": No time mentioned or non-meeting

CRITICAL REQUIREMENTS:
- ALWAYS return valid JSON
- NEVER add explanations before/after JSON
- Extract ALL email addresses found (don't miss any attendees)
- Convert times to UTC ISO 8601 format consistently
- For non-meetings: set is_meeting=false and all other fields to null or empty array"""


In [ ]:
# =============================================================================
# STEP 1: LOAD AND PREPARE DATA
# =============================================================================

def load_training_data():
    """Load and format training data with optimized system prompt"""

    print("\n" + "=" * 80)
    print("STEP 1: Loading Training Data")
    print("=" * 80)

    # Check if data files exist
    for filepath in [TRAIN_FILE, VAL_FILE]:
        if not os.path.exists(filepath):
            raise FileNotFoundError(
                f"\n Data file not found: {filepath}\n"
                f"\nIf using Google Drive, make sure to:"
                f"\n  1. Create folder: {BASE_DATA_DIR}"
                f"\n  2. Upload train.jsonl, val.jsonl, test.jsonl to that folder"
                f"\n  3. Check the folder exists in Drive"
            )

    def load_jsonl(filepath):
        data = []
        with open(filepath, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        return data

    train_data = load_jsonl(TRAIN_FILE)
    val_data = load_jsonl(VAL_FILE)

    print(f"\n Loaded training data:")
    print(f"   Train: {len(train_data):,} examples")
    print(f"   Val:   {len(val_data):,} examples")

    # Convert to Qwen chat format with production system prompt
    def convert_to_qwen_format(examples):
        formatted = []

        for ex in examples:
            original_messages = ex['messages']

            # Replace system prompt with production-optimized version
            new_messages = [
                {"role": "system", "content": PRODUCTION_SYSTEM_PROMPT},
                original_messages[1],  # User message (email)
                original_messages[2]   # Assistant message (extraction)
            ]

            formatted.append({
                "messages": new_messages,
                "text": ""
            })

        return formatted

    print("\n Applying production-optimized system prompt...")
    train_formatted = convert_to_qwen_format(train_data)
    val_formatted = convert_to_qwen_format(val_data)

    print(f"Replaced system prompt in all {len(train_formatted) + len(val_formatted):,} examples")

    # Create HuggingFace datasets
    train_dataset = Dataset.from_list(train_formatted)
    val_dataset = Dataset.from_list(val_formatted)

    return train_dataset, val_dataset

train_data, val_data = load_training_data()




STEP 1: Loading Training Data

 Loaded training data:
   Train: 12,753 examples
   Val:   708 examples

 Applying production-optimized system prompt...
Replaced system prompt in all 13,461 examples


In [ ]:
# =============================================================================
# STEP 2: LOAD MODEL WITH UNSLOTH
# =============================================================================

def load_model_and_tokenizer():
    """Load Qwen 2.5 14B with Unsloth optimizations"""

    print("\n" + "=" * 80)
    print("STEP 2: Loading Model with Unsloth")
    print("=" * 80)

    print(f"\n1. Loading {MODEL_NAME}...")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
        token=HF_TOKEN,
    )

    print(f"Model loaded successfully")

    # Add LoRA adapters
    print(f"\n2. Adding LoRA adapters...")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"\n Model Statistics:")
    print(f"   Trainable: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    print(f"   Total: {total_params:,}")

    return model, tokenizer
print("\n Loading model (this may take 2-3 minutes)...")
model, tokenizer = load_model_and_tokenizer()
print("Model ready!")


 Loading model (this may take 2-3 minutes)...

STEP 2: Loading Model with Unsloth

1. Loading Qwen/Qwen2.5-14B-Instruct...


2026-02-22 05:31:47,660 - modelscope - INFO - Got 14 files, start to download ...


Processing 14 items:   0%|          | 0.00/14.0 [00:00<?, ?it/s]

2026-02-22 06:26:53,619 - modelscope - INFO - Download model 'unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit' successfully.
2026-02-22 06:26:53,620 - modelscope - INFO - Creating symbolic link [/root/.cache/modelscope/hub/models/unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit].


==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Model loaded successfully

2. Adding LoRA adapters...


Unsloth 2026.2.1 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



 Model Statistics:
   Trainable: 68,812,800 (0.79%)
   Total: 8,688,948,224
Model ready!


In [ ]:
# =============================================================================
# step 3: PREPARE DATASETS
# =============================================================================

def prepare_datasets(train_data, val_data, tokenizer):
    """Prepare datasets for training"""

    print("\n" + "=" * 80)
    print("PREPARING DATASETS")
    print("=" * 80)

    # Format function - MUST be defined here with tokenizer in scope
    def formatting_func(examples):
        """Format examples using chat template"""
        texts = []
        for messages in examples["messages"]:
            # Apply Qwen chat template
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
        return texts

    print("\n Converting to HuggingFace datasets...")

    # Create datasets
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    # Apply formatting - THIS IS KEY!
    print(" Applying chat template formatting...")

    train_dataset = train_dataset.map(
        lambda x: {"text": formatting_func(x)},
        batched=True,
        remove_columns=train_dataset.column_names
    )

    val_dataset = val_dataset.map(
        lambda x: {"text": formatting_func(x)},
        batched=True,
        remove_columns=val_dataset.column_names
    )

    print(f"\nDatasets prepared:")
    print(f"   Train: {len(train_dataset)} examples")
    print(f"   Val:   {len(val_dataset)} examples")

    # Show sample
    print(f"\n Sample formatted text (first 300 chars):")
    print(train_dataset[0]['text'][:300] + "...")

    return train_dataset, val_dataset

# Prepare datasets
train_dataset, val_dataset = prepare_datasets(train_data, val_data, tokenizer)




PREPARING DATASETS

 Converting to HuggingFace datasets...
 Applying chat template formatting...


Map:   0%|          | 0/12753 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]


Datasets prepared:
   Train: 12753 examples
   Val:   708 examples

 Sample formatted text (first 300 chars):
<|im_start|>system
You are a specialized meeting information extraction system.

TASK: Analyze emails and extract structured meeting details with high precision.

OUTPUT FORMAT (JSON only, no explanations):
{
  "is_meeting": boolean,
  "title": string or null,
  "attendees": array of email addresses...


In [ ]:
# =============================================================================
# step 4: SETUP TRAINER
# =============================================================================

def setup_trainer(model, tokenizer, train_dataset, val_dataset):
    """Setup trainer"""

    print("\n" + "=" * 80)
    print("SETTING UP TRAINER")
    print("=" * 80)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=SAVE_STEPS,
        eval_strategy="steps",  # Changed from evaluation_strategy
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field="text",  # Use the "text" column we created
        max_seq_length=MAX_SEQ_LENGTH,
        args=training_args,
    )

    total_steps = len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION) * NUM_EPOCHS

    print(f"\nTrainer ready")
    print(f"   Total steps: {total_steps}")
    print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

    return trainer

# Setup trainer
trainer = setup_trainer(model, tokenizer, train_dataset, val_dataset)



SETTING UP TRAINER


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/12753 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/708 [00:00<?, ? examples/s]


Trainer ready
   Total steps: 4782
   Effective batch: 8


In [ ]:
# =============================================================================
# step 5: TRAIN MODEL
# =============================================================================

def train_model(trainer, output_dir=OUTPUT_DIR):
    """Train the model (auto-resumes from checkpoint if exists)"""

    print("\n" + "=" * 80)
    print("TRAINING MODEL")
    print("=" * 80)

    # Check for existing checkpoints
    import glob
    checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
    checkpoints.sort()

    if checkpoints:
        latest_checkpoint = checkpoints[-1]
        checkpoint_step = latest_checkpoint.split('-')[-1]

        print(f"\nFound {len(checkpoints)} existing checkpoint(s)")
        print(f" Latest checkpoint: {latest_checkpoint}")
        print(f"Resuming from step: {checkpoint_step}")
        print()

        # Enable training mode
        FastLanguageModel.for_training(trainer.model)

        # Resume from checkpoint
        trainer.train(resume_from_checkpoint=latest_checkpoint)

    else:
        print("\n No checkpoints found - starting fresh training")
        print(" This will take 3-4 hours on A100")
        print(" Checkpoints save to Google Drive")
        print()

        # Enable training mode
        FastLanguageModel.for_training(trainer.model)

        # Start fresh training
        trainer.train()

    print("\n Training complete!")

    return trainer


In [ ]:
trainer = train_model(trainer)


TRAINING MODEL

Found 3 existing checkpoint(s)
 Latest checkpoint: /content/drive/MyDrive/qwen-meeting-extraction/checkpoint-950
Resuming from step: 950



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,753 | Num Epochs = 3 | Total steps = 4,785
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)


Step,Training Loss,Validation Loss
1000,0.223400,0.239815
1050,0.260600,0.238659
1100,0.219700,0.238483
1150,0.221500,0.237876
1200,0.223600,0.236860
1250,0.241800,0.236633
1300,0.238700,0.235952
1350,0.234300,0.235610
1400,0.226900,0.235214
1450,0.234600,0.234731


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



 Training complete!


In [ ]:
# =============================================================================
# step 6: SAVE MODEL (Run after training completes)
# =============================================================================

def save_model(model, tokenizer):
    """Save the fine-tuned model"""

    print("\n" + "=" * 80)
    print("SAVING MODEL")
    print("=" * 80)

    # Save LoRA adapters
    lora_path = f"{OUTPUT_DIR}/lora_adapters"
    model.save_pretrained(lora_path)
    tokenizer.save_pretrained(lora_path)
    print(f"\n LoRA adapters: {lora_path}")

    # Save system prompt
    with open(f"{OUTPUT_DIR}/system_prompt.txt", 'w') as f:
        f.write(PRODUCTION_SYSTEM_PROMPT)
    print(f" System prompt saved")

    # Save merged model
    print("\n Saving merged model (this takes a few minutes)...")
    merged_path = f"{OUTPUT_DIR}/merged_model"
    model.save_pretrained_merged(
        merged_path,
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"Merged model: {merged_path}")

    if IS_COLAB:
        print(f"\n All saved to Google Drive!")
        print(f"   {OUTPUT_DIR}")

    print("\n COMPLETE!")

# After training, run:
save_model(model, tokenizer)